Import Libraries 

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

Build the modeling table

In [3]:
current = Path.cwd().resolve()
project_root = next(
    path for path in (current, *current.parents)
    if (path / "pyproject.toml").is_file()
)
raw_path = project_root / "data" / "raw" / "N45B-nmos1.json"

#open the raw data file and load the JSON content
with raw_path.open(encoding="utf-8") as file:
    mesd = json.load(file)

columns = ["W", "L", "Temp", "Corner", "Vds", "Vgs", "Ids"]


points = (
    pd.DataFrame(mesd["Records"])[columns]
    .rename_axis("sweep_id")
    .explode(["Vgs", "Ids"])
    .reset_index()
    .astype({"Vgs": float, "Ids": float})
)

df = points.loc[points["Vds"] > 0].copy().reset_index(drop=True)
assert np.isfinite(df["Ids"]).all() and df["Ids"].gt(0).all()

df["log10_Ids"] = np.log10(df["Ids"])

print(f"Raw operating points: {len(points):,}")
print(f"Positive-drain-bias points: {len(df):,}")
print(f"Positive-drain-bias sweeps: {df['sweep_id'].nunique():,}")
print(f"log10(Ids) range: {df['log10_Ids'].min():.2f} to {df['log10_Ids'].max():.2f}")
display(df.head())

Raw operating points: 99,000
Positive-drain-bias points: 82,500
Positive-drain-bias sweeps: 7,500
log10(Ids) range: -12.39 to -3.25


,sweep_id,W,L,Temp,Corner,Vds,Vgs,Ids,log10_Ids
0,1,120,60,-40,tt,0.2,0.0,4.710100e-13,-12.326970
1,1,120,60,-40,tt,0.2,0.1,3.849100e-12,-11.414641
2,1,120,60,-40,tt,0.2,0.2,1.758100e-10,-9.754956
3,1,120,60,-40,tt,0.2,0.3,5.683300e-09,-8.245399
4,1,120,60,-40,tt,0.2,0.4,6.878800e-08,-7.162487


Fix and verify the held-out-width split

In [4]:
#split the dataset into training and testing sets based on unique widths
test_widths = [243, 624]
assert set(test_widths).issubset(df["W"].unique())

df["split"] = np.where(df["W"].isin(test_widths), "test", "train")

train = df.loc[df["split"].eq("train")]
test = df.loc[df["split"].eq("test")]

assert set(train["W"]).isdisjoint(test["W"])
assert set(train["sweep_id"]).isdisjoint(test["sweep_id"])

display(
    df.groupby("split").agg(
        rows=("Ids", "size"),
        sweeps=("sweep_id", "nunique"),
        widths=("W", "nunique"),
    )
)

#save the processed dataset to a CSV file
processed_path = project_root / "data" / "processed" / "n45b_nmos1_iv.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(processed_path, index=False)

,rows,sweeps,widths
split,,,
test,16500,1500,2
train,66000,6000,8
